In [ ]:
module chebyshev_method

    using LinearAlgebra

    export chebyshev_D
    # Chebyshev compute D = differentiation matrix, x = Chebyshev grid

    function  chebyshev_D(N)

        if N==0
            D = 0;
            x = 1;
            return D, x
        else
            θ = range(0,length=N+1,stop=pi)
            x = reshape(-cos.(θ), N+1, 1)
            c = [2; ones(N-1, 1) ; 2] .* (-1) .^ (0:N)
            X = repeat(x, 1, N+1);
            dX = X - X';
            D = (c * (1 ./ c)') ./ (dX .+ I(N+1));   # off-diagonal entries
            D = D - diagm(vec(sum(D, dims=2)));      # diagonal entries
            return D, x
        end
    end
end

In [ ]:
function rayleigh_quotient_iteration(A, B, sigma; q0=rand(size(A, 1), 1))

    flg = true
    while flg
        sigma0 = sigma[1] + 0.0e0im
        q = (A - sigma*B) \ (B*q0)
        q0 = q/maximum(abs.(q))
        sigma = ((q0'*(A*q0))/(q0'*(B*q0)))[1]
        if abs(sigma-sigma0)<=eps(1.0f0)
            flg = false
        end
    end

    return sigma, q0

end

In [ ]:
#interpolate data
using LinearAlgebra
using BSplineKit
    f = open( "Vonkarmen copy 2.txt", "r" )
    n = countlines( f )
    seekstart( f )
    data = zeros(n,6)
    for i = 1:n
        z,w,u,v,du,dv = split( readline( f ), " " )  # 读取每一行数据并用split函数将数据“剥离”开来
        data[i,1] = parse(Float64,z)
        data[i,2] = parse(Float64,w)
        data[i,3] = parse(Float64,u)
        data[i,4] = parse(Float64,v)  # 将字符串转化为数字
        data[i,5] = parse(Float64,du)
        data[i,6] = parse(Float64,dv)
    end

    close( f )

    t=data[:,1]
    w=data[:,2]
    u=data[:,3]
    v=data[:,4]
    du=data[:,5]
    dv=data[:,6]
    itpw=itp = interpolate(t, w , BSplineOrder(4))
    itpu=itp = interpolate(t, u , BSplineOrder(4))
    itpv=itp = interpolate(t, v , BSplineOrder(4))
    itpdu=itp = interpolate(t, du , BSplineOrder(4))
    itpdv=itp = interpolate(t, dv , BSplineOrder(4));

In [44]:
using .chebyshev_method
    N=99
    D,x=chebyshev_D(N)
    # for i=1:N+1
    #     D[i,:]=D[i,:].*((2*x[i]^3-x[i]^2+3*x[i]-4)^2/(20*(6*x[i]^2-2*x[i]+3)))
    # end
    # for i=1:N+1
    #     x[i]=(4*x[i]^3-2*x[i]^2+6*x[i]+12)/(-2*x[i]^3+x[i]^2-3*x[i]+4)
    #     if x[i]>200
    #         x[i]=200
    #     end
    # end
    for i=1:N+1
        x[i,1]=100* x[i,1]+ 100
    end
    D=0.01* D
    D2=D^2;

In [ ]:
    α=0.38482
    β=0.07759
    R=285.36
    U=zeros(N+1,1)
    V=zeros(N+1,1)
    W=zeros(N+1,1)
    dU=zeros(N+1,1)
    dV=zeros(N+1,1)
    dW=zeros(N+1,1)
    p=zeros(N+1,1)
    for i=1:N+1
        U[i,1]=itpu(x[i])
        V[i,1]=itpv(x[i])
        W[i,1]=itpw(x[i])
        dU[i,1]=itpdu(x[i])
        dV[i,1]=itpdv(x[i])
    end
    dW=-2*U
    ddU=D*dU;
    ddV=D*dV;
    α_bar=α-im/R
    λ=sqrt(α^2+β^2)
    λ_bar=sqrt(α*α_bar+β^2)
    A11=im*((D2-(λ^2)*I(N+1))*(D2-(λ_bar^2)*I(N+1)))+R*((α*U+β*V)).*(D2-(λ_bar^2)*I(N+1))-R*(α_bar*ddU+β*ddV).*I(N+1)-im*W.*D*(D^2-(λ_bar^2)*I(N+1))-im*dW.*(D2-(λ_bar^2)*I(N+1))-im*U.*D2
    A12=2*(V).*D+2*dV.*I(N+1)
    A21=2*(V).*D-im*R*(α*dV-β*dU).*I(N+1)
    A22=im*(D2-(λ^2)*I(N+1))+R*(α*U+β*V).*I(N+1)-im*W.*D-im*U.*I(N+1)
    B11=R*(D2-(λ_bar^2)*I(N+1))
    B12=zeros(N+1,N+1)
    B21=B12
    B22=R*I(N+1)
    B22=Matrix(B22)
    #boundary condition
    A11[2,:]=D[1,:]
    A12[2,:].=0
    A21[N,:]=D[N+1,:]
    A22[N,:].=0
    B11[2,:]=B12[2,:].=0
    B21[N,:]=B22[N,:].=0
    A11=A11[2:N,2:N]
    A12=A12[2:N,2:N]
    A21=A21[2:N,2:N]
    A22=A22[2:N,2:N]
    B11=B11[2:N,2:N]
    B12=B12[2:N,2:N]
    B21=B21[2:N,2:N]
    B22=B22[2:N,2:N]
    A=[A11 A12;A21 A22];
    B=[B11 B12;B21 B22];
    A=A[1:195,1:195];
    B=B[1:195,1:195];

In [ ]:
@time C=eigen(A,B)
values=C.values
vectors=C.vectors
K=filter(x-> 0<imag(x)<1,values)
K1=map(x->0<imag(x)<1,values)
K2=findmax(K1)
index=K2[2]
c_temp=values[index,1]
q1=vectors[:,index]
@time c,q=rayleigh_quotient_iteration(A,B,c_temp,q0=q1)
c

In [45]:
ω=-0.0039825170752425245
A=D2-(β^2)*I(N+1)
L0=zeros(2*(N+1),2*(N+1));
L0_1=im*A^2+R*β*V.*A-R*ω.*A-R*β*ddV.*I(N+1)+im*ddU.*I(N+1)-im*W.*(D*A)-im*dW.*A-im*U.*D^2
L0_2=2*V.*D+2*dV.*I(N+1)
L0_3=2*V.*D+im*R*β*dU.*I(N+1)
L0_4=im*A+R*β*V.*I(N+1)-R*ω*I(N+1)-im*W.*D-im*U.*I(N+1)
L1_1=-(1/R).*A-R*U.*A+im*β*V.*I(N+1)-im*ω*I(N+1)-R*ddU.*I(N+1)+(1/R)*W.*D+(1/R)*dW.*I(N+1)
L1_2=zeros(N+1,N+1)
L1_3=-1*im*R*dV.*I(N+1)
L1_4=R*U.*I(N+1)
L2_1=-2*im*A+ω*R*I(N+1)-β*R*V.*I(N+1)+im*U.*I(N+1)+im*W.*D+im*dW.*I(N+1)
L2_2=L1_2
L2_3=L1_2
L2_4=-im*I(N+1)
L3_1=(1/R)*I(N+1)-R*U.*I(N+1)
L3_2=L3_3=L3_4=L1_2
L4_1=im*I(N+1)
L4_2=L4_3=L4_4=L1_2
L2_4=Matrix{ComplexF64}(L2_4)
#boundary condition
L0_1[2,:]=L0_2[2,:]=L1_1[2,:]=L1_2[2,:]=L2_1[2,:]=L2_2[2,:]=L3_1[2,:]=L3_2[2,:].=0
L0_1[2,:]=D[1,:]

L0_3[N,:]=L0_4[N,:]=L1_3[N,:]=L1_4[N,:]=L2_3[N,:]=L2_4[N,:]=L3_3[N,:]=L3_4[N,:].=0
L0_3[N]=D[N+1]

L0_1[3,:]=L0_2[3,:]=L1_1[3,:]=L1_2[3,:]=L2_1[3,:]=L2_2[3,:]=L3_1[3,:]=L3_2[3,:].=0
L0_2[3,:]=D[1,:]

L0_3[N-1,:]=L0_4[N-1,:]=L1_3[N-1,:]=L1_4[N-1,:]=L2_3[N-1,:]=L2_4[N-1,:]=L3_3[N-1,:]=L3_4[N-1,:].=0
L0_4[N-1]=D[N+1]

L0_1[4,:]=L0_2[4,:]=L1_1[4,:]=L1_2[4,:]=L2_1[4,:]=L2_2[4,:]=L3_1[4,:]=L3_2[4,:].=0
L1_1[4,:]=D[1,:]

L0_3[N-2,:]=L0_4[N-2,:]=L1_3[N-2,:]=L1_4[N-2,:]=L2_3[N-2,:]=L2_4[N-2,:]=L3_3[N-2,:]=L3_4[N-2,:].=0
L1_3[N-2]=D[N+1]

L0_1[5,:]=L0_2[5,:]=L1_1[5,:]=L1_2[5,:]=L2_1[5,:]=L2_2[5,:]=L3_1[5,:]=L3_2[5,:].=0
L1_2[5,:]=D[1,:]

L0_3[N-3,:]=L0_4[N-3,:]=L1_3[N-3,:]=L1_4[N-3,:]=L2_3[N-3,:]=L2_4[N-3,:]=L3_3[N-3,:]=L3_4[N-3,:].=0
L1_4[N-2]=D[N+1]

L0_1[6,:]=L0_2[6,:]=L1_1[6,:]=L1_2[6,:]=L2_1[6,:]=L2_2[6,:]=L3_1[6,:]=L3_2[6,:].=0
L2_1[6,:]=D[1,:]

L0_3[N-4,:]=L0_4[N-4,:]=L1_3[N-4,:]=L1_4[N-4,:]=L2_3[N-4,:]=L2_4[N-4,:]=L3_3[N-4,:]=L3_4[N-4,:].=0
L2_3[N-4]=D[N+1]

L0_1[7,:]=L0_2[7,:]=L1_1[7,:]=L1_2[7,:]=L2_1[7,:]=L2_2[7,:]=L3_1[7,:]=L3_2[7,:].=0
L2_2[7,:]=D[1,:]

L0_3[N-5,:]=L0_4[N-5,:]=L1_3[N-5,:]=L1_4[N-5,:]=L2_3[N-5,:]=L2_4[N-5,:]=L3_3[N-5,:]=L3_4[N-5,:].=0
L2_4[N-5]=D[N+1]

L0_1[8,:]=L0_2[8,:]=L1_1[8,:]=L1_2[8,:]=L2_1[8,:]=L2_2[8,:]=L3_1[8,:]=L3_2[8,:].=0
L3_1[8,:]=D[1,:]

L0_3[N-6,:]=L0_4[N-6,:]=L1_3[N-6,:]=L1_4[N-6,:]=L2_3[N-6,:]=L2_4[N-6,:]=L3_3[N-6,:]=L3_4[N-6,:].=0
L3_3[N-6]=D[N+1]

L0_1[9,:]=L0_2[9,:]=L1_1[9,:]=L1_2[9,:]=L2_1[9,:]=L2_2[9,:]=L3_1[9,:]=L3_2[9,:].=0
L3_2[9,:]=D[1,:]

L0_3[N-7,:]=L0_4[N-7,:]=L1_3[N-7,:]=L1_4[N-7,:]=L2_3[N-7,:]=L2_4[N-7,:]=L3_3[N-7,:]=L3_4[N-7,:].=0
L3_4[N-7]=D[N+1]

L0_1=L0_1[2:end-1,2:end-1]
L0_2=L0_2[2:end-1,2:end-1]
L0_3=L0_3[2:end-1,2:end-1]
L0_4=L0_4[2:end-1,2:end-1]
L1_1=L1_1[2:end-1,2:end-1]
L1_2=L1_2[2:end-1,2:end-1]
L1_3=L1_3[2:end-1,2:end-1]
L1_4=L1_4[2:end-1,2:end-1]
L2_1=L2_1[2:end-1,2:end-1]
L2_2=L2_2[2:end-1,2:end-1]
L2_3=L2_3[2:end-1,2:end-1]
L2_4=L2_4[2:end-1,2:end-1]
L3_1=L3_1[2:end-1,2:end-1]
L3_2=L3_2[2:end-1,2:end-1]
L3_3=L3_3[2:end-1,2:end-1]
L3_4=L3_4[2:end-1,2:end-1]
L4_1=L4_1[2:end-1,2:end-1]
L4_2=L4_2[2:end-1,2:end-1]
L4_3=L4_3[2:end-1,2:end-1]
L4_4=L4_4[2:end-1,2:end-1]
L0=[L0_1 L0_2;L0_3 L0_4]
L1=[L1_1 L1_2;L1_3 L1_4]
L2=[L2_1 L2_2;L2_3 L2_4]
L3=[L3_1 L3_2;L3_3 L3_4]
L4=[L4_1 L4_2;L4_3 L4_4]
L5=zeros(2*(N-1),2*(N-1))
L6=I(2*(N-1))
A=[L0 L1 L2 L3;L5 L6 L5 L5;L5 L5 L6 L5;L5 L5 L5 L6]
B=[L5 L5 L5 -1*L4;L6 L5 L5 L5;L5 L6 L5 L5;L5 L5 L6 L5]

B[1:8,:].=0
B[N-8:N-1,:].=0

8×784 view(::Matrix{ComplexF64}, 91:98, :) with eltype ComplexF64:
 0.0+0.0im  0.0+0.0im  0.0+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  …  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im     0.0+0.0im  0.0+0.0im  0.0+0.0im

In [46]:
@time C=eigen(A,B)

  5.141763 seconds (19 allocations: 28.646 MiB)


GeneralizedEigen{ComplexF64, ComplexF64, Matrix{ComplexF64}, Vector{ComplexF64}}
values:
784-element Vector{ComplexF64}:
  -8.228067314362464e33 + 5.380035776419782e32im
 -2.1943213344706994e30 + 8.390082561417801e29im
  -9.978285329265415e29 + 1.885980221707801e30im
  -8.878752106456284e29 - 1.4188372433935098e30im
   -6.07461435450649e29 + 3.7809932540963244e29im
 -4.7413387633776525e29 - 3.378729378140066e29im
 -1.1455824704991232e29 - 3.9065467524616046e28im
 -1.0623082143642832e29 + 1.6287110360745816e30im
   -8.17011999912904e28 - 2.6112663481137876e29im
  -3.074403326315618e28 - 3.0954840071197602e28im
                        ⋮
                    NaN + NaN*im
                    NaN + NaN*im
                    NaN + NaN*im
                    NaN + NaN*im
                    NaN + NaN*im
                    NaN + NaN*im
                    NaN + NaN*im
                    NaN + NaN*im
                    NaN + NaN*im
vectors:
784×784 Matrix{ComplexF64}:
  -1.92227e-29-5.34741e

In [47]:
values=C.values
vectors=C.vectors
K=filter(x-> -1<imag(x)<0,values)
# K1=map(x->0<imag(x)<1,values)
# K2=findmax(K1)
# index=K2[2]
# c_temp=values[index,1]
# q1=vectors[:,index]
# @time c,q=rayleigh_quotient_iteration(A,B,c_temp,q0=q1)
# c

104-element Vector{ComplexF64}:
  -1.9226815172740956 - 0.9031909850049032im
 -0.34923649801620743 - 0.18240534328429783im
 -0.16540097681870392 - 0.12007841028924816im
   -0.158839835644893 - 0.900882491237006im
 -0.15784568146229147 - 0.23403267213900159im
 -0.15594226359367472 - 0.07236508310386541im
 -0.15052970750393413 - 0.29624768942202745im
 -0.14771430330576765 - 0.7767157838592983im
 -0.13554298764347916 - 0.6793215938530625im
 -0.13452030328854464 - 0.46334209503609963im
                      ⋮
   0.7015137524332958 - 0.785831897493499im
   0.7050495247124922 - 0.807384761027469im
   0.7261671799900601 - 0.00455130051217272im
   0.7489609432574886 - 0.7408615413170494im
   0.7594271538239734 - 0.7485089399508015im
   0.7864851360611206 - 0.7840201500304609im
   0.8071333045311783 - 0.7046224184655402im
   1.3323066218771074 - 0.4323972876325265im
    2.893465760663514 - 0.8579142630383328im